# Données sur les faits divers

### Notes :
Erreur 403 sur le site de Legifrance -> nécessicité de passer par l'API, avec une inscription --> barbant

Pistes possibles : utiliser des données sur les pages Wikipedia suivantes :

https://fr.wikipedia.org/wiki/Enqu%C3%AAtes_criminelles_:_le_magazine_des_faits_divers

Épisodes d'enquêtes sur des affaires criminelles -> à faire : trier les faits divers français et étrangers

https://fr.wikipedia.org/wiki/Liste_d'affaires_criminelles_fran%C3%A7aises_depuis_1900

Faits divers recensés sur Wikipedia\
Nature des données :\
tableaux triés par décénies\
plage de données : 1900 à 2020\
Colonnes d'intérêt : Départements et résumés\
Départements : propres\
Résumés : à extraire : les dates (JJ mois AAAA) \
Limite : gros faits divers médiatisés, probablement aucun rapport avec la météo...



## Récupération et nettoyage des données de la page Wikipédia sur les affaires criminelles

### Récupération des données

In [1]:
### Récupération des données sur la page Wikipédia

import requests

url = "https://fr.wikipedia.org/wiki/Liste_d'affaires_criminelles_fran%C3%A7aises_depuis_1900"

req = requests.get(url, headers={"User-Agent":"School Data Science Project"})
print(req)

<Response [200]>


In [2]:
### Transformation en fichier exploitable
import bs4

request_text = req.content
page = bs4.BeautifulSoup(request_text, "lxml")
print(page)

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-underline-clientpref-browser" dir="ltr" lang="fr">
<head>
<meta charset="utf-8"/>
<title>Liste d'affaires criminelles françaises depuis 1900 — Wikipédia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-featur

### Récupération des tableaux

In [3]:
### On récupère tous les tableaux de la page, on triera plus tard selon les dates

tableaux = page.find_all('table', {'class':'wikitable sortable'})
print(len(tableaux))

24


In [4]:
### Transformation en listes de string pour manipuler les tableaux

faitsDiv = list()
for tableau in tableaux :
    table_body = tableau.find('tbody')
    rows = table_body.find_all('tr')
    liste = list()
    for row in rows:
        cols = row.find_all('td')
        cols = [ele.text.strip() for ele in cols]
        if len(cols) > 0 : 
            liste.append(cols)
    faitsDiv.append(liste)

faitsDiv

[[['1905 - 1908',
   'Drôme',
   'Chauffeurs de la Drôme',
   "De 1905 à 1908 dans la Drôme, actes de brigandage suivis d'assassinats par Urbain Liottard, Octave David et Louis Berruyer.",
   'Urbain Liottard est condamné le 10 juillet 1909 à la peine de mort, Octave David à la peine de mort, Louis Berruyer à la peine de mort par les assises de la Drôme. Guillotinés le 22 septembre 1909 à Valence.'],
  ['1907',
   'Seine',
   'Affaire Soleilland',
   'Le 31 janvier 1907 à Paris, meurtre de Marthe Erbelding, onze ans, après avoir été violée et poignardée.',
   'Albert Soleilland est condamné le 23 juillet 1907 à la peine de mort. Gracié, il meurt au bagne en 1920 après 13 ans de détention.'],
  ['1907',
   'Gironde',
   'Crime de Langon',
   "Le 6 février 1907 à Langon, assassinat crapuleux d'un agent d'assurance, Jean-Théodore Monget étranglé avec une serviette, commis par Jean-Eugène Branchery et Henri Parrot.",
   'Jean-Eugène Branchery est condamné le 29 février 1908 à la peine de m

In [5]:
### On transforme les listes en DataFrames
import pandas as pd

liste_dataf = list()
for tableau in faitsDiv :
    liste_dataf.append(pd.DataFrame(tableau))



In [7]:
### On ajoute des titres aux colonnes

## On récupère les noms de colonnes

liste_colonnes = list()

for tableau in tableaux :
    table_body = tableau.find('tbody')
    rows = table_body.find_all('tr')
    for row in rows:
        cols = row.find_all('th')
        if len(cols) > 0 : 
            cols = [ele.get_text(separator=' ').strip().title() for ele in cols]
            liste_colonnes.append(cols)

## On rajoute les colonnes sur le data frame :

for i in range(len(liste_dataf)):
    liste_dataf[i].columns = liste_colonnes[i]

liste_dataf[5].head()


,Année [ Note 1 ],Département(S) [ Note 2 ],Affaire,Résumé,Condamnations Et Commentaires
0,1951,Guadeloupe,Affaire Fengarol,"Assassinat supposé de Amédée Fengarol, 45 ans,...",
1,1951,Paris,Affaire Dubuisson,Assassinat de Félix Bailly par son ex-amoureus...,Cette affaire a inspiré le personnage du film ...
2,1951,Loiret,Affaire Chevallier,"Assassinat, de Pierre Chevallier député et sec...",
3,1952,Alpes-de-Haute-Provence,Affaire Dominici,Meurtre de trois membres d'une famille anglais...,L'affaire inspire notamment L'Affaire Dominici...
4,1953,Pyrénées-Orientales,Affaire Marty,Empoisonnement supposé au gardénal de Jeanne C...,


In [13]:
### On concatène tout pour faire un gros tableau

faitsDivDF = pd.concat(liste_dataf)
faitsDivDF = faitsDivDF.reset_index(drop=True)

print(len(faitsDivDF))
faitsDivDF.head(15)

474


,Année [ Note 1 ],Département(S) [ Note 2 ],Affaire,Résumé,Condamnations Et Commentaires,Commentaires Et Condamnations
0,1905 - 1908,Drôme,Chauffeurs de la Drôme,"De 1905 à 1908 dans la Drôme, actes de brigand...",Urbain Liottard est condamné le 10 juillet 190...,NaN
1,1907,Seine,Affaire Soleilland,"Le 31 janvier 1907 à Paris, meurtre de Marthe ...",Albert Soleilland est condamné le 23 juillet 1...,NaN
2,1907,Gironde,Crime de Langon,"Le 6 février 1907 à Langon, assassinat crapule...",Jean-Eugène Branchery est condamné le 29 févri...,NaN
3,1909,Seine-et-Oise,Affaire Bourette,"Le 22 octobre 1909 au Vésinet, empoisonnement ...",Marie Bourette est condamnée le 13 juillet 191...,NaN
4,1910,Seine,Affaire Liabeuf,"Le 8 janvier 1910 à Paris, assassinat d'un pol...",Jean-Jacques Liabeuf est condamné le 4 mai 191...,NaN
5,1910,Seine-Inférieure,Affaire Durand,"Le 9 septembre 1910 au Havre, meurtre de Louis...",Jules Durand est condamné le 25 novembre 1910 ...,NaN
6,1911-1912,"Paris, Val-de-Marne, Val-d'Oise, Oise",La Bande à Bonnot,Série de braquages et de meurtres.,Les méfaits de la bande inspirent notamment un...,NaN
7,1913,Loire-Inférieure,Affaire Marcel Redureau,Meurtre de sept personnes le 30 septembre 1913...,"Condamné à 20 ans de prison, Marcel Redureau m...",NaN
8,1914,Finistère,Affaire Cadiou,"Le 4 février 1914, le corps de Louis Cadiou es...","Jugé cinq ans après les faits, l'ingénieur Lou...",NaN
9,1914,Paris,Affaire Caillaux,"Assassinat du directeur du Figaro, Gaston Calm...",Le président de la République fait une déposit...,NaN
